In [ ]:
# IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb

### LOAD DATA

In [3]:
df_0 = pd.read_excel('../data/raw/PARATEC_Capacidadefectiva_27-10-2025.xlsx', sheet_name='CapacidadEfectiva_2', skiprows=7)

### DATA CLEANING

In [ ]:
# 1. Remove rows with no data

ultimas_5 = df_0.columns[-5:]  # Select the last 5 columns
df = df_0.dropna(subset=ultimas_5, how='all')

# 2. Rename column to a more descriptive name
df = df.rename(columns={'Tipo/Fuente de energía': 'NombrePlanta'})

# 3. Convert string data to numeric
df['Capacidad efectiva neta [MW]'] = (
    df['Capacidad efectiva neta [MW]']
    .astype(str)
    .str.replace('.', '', regex=False)   # Remove thousands separator
    .str.replace(',', '.', regex=False)  # Change decimal separator
    .astype(float)
)

# 4. Convert text to date with day/month/year format
df['Fecha de puesta en operación FPO'] = pd.to_datetime(df['Fecha de puesta en operación FPO'], format='%d/%m/%Y', errors='coerce')

In [6]:
df.columns

Index(['NombrePlanta', 'FUENTE DE ENERGIA', 'Despacho', 'Generador',
       'Capacidad efectiva neta [MW]', 'Operador',
       'Fecha de puesta en operación FPO', 'Municipio', 'Departamento',
       'Subárea'],
      dtype='object')

### CALCULATE CUMULATIVE CAPACITY

In [ ]:
df_series = df[['Fecha de puesta en operación FPO', 'Capacidad efectiva neta [MW]', 'FUENTE DE ENERGIA']].copy()

df_series = df_series.sort_values(by='Fecha de puesta en operación FPO', ascending=True).reset_index(drop=True)

# Create cumulative capacity column
df_series['Capacidad acumulada [MW]'] = df_series['Capacidad efectiva neta [MW]'].cumsum()

# ==============================
# 1. FIGURE: df_series
# ==============================
plt.figure(figsize=(5,3))
plt.plot(df_series['Fecha de puesta en operación FPO'],
         df_series['Capacidad acumulada [MW]'])
plt.xlabel('Date')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Cumulative capacity growth (df_series)')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

# Iterate through each energy source category and plot with a label
for source, df_sub in df_series.groupby('FUENTE DE ENERGIA'):
    plt.scatter(df_sub['Fecha de puesta en operación FPO'],
                df_sub['Capacidad acumulada [MW]'],
                label=source)

plt.xlabel('Commissioning date (FPO)')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Cumulative capacity by energy source type')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.legend(title='Energy source')
plt.tight_layout()
plt.show()

In [ ]:
# Sort by date to ensure correct accumulation
df_cumul_source = df_series.sort_values(by='Fecha de puesta en operación FPO').copy()

# Cumulative by each energy source
df_cumul_source['Capacidad acumulada por fuente [MW]'] = (
    df_cumul_source
    .groupby('FUENTE DE ENERGIA')['Capacidad efectiva neta [MW]']
    .cumsum())

plt.figure(figsize=(7,3))

# Iterate through each energy source and plot its cumulative curve
for source, df_sub in df_cumul_source.groupby('FUENTE DE ENERGIA'):
    plt.plot(df_sub['Fecha de puesta en operación FPO'],
             df_sub['Capacidad acumulada por fuente [MW]'],
             label=source)

plt.xlabel('Commissioning date')
plt.ylabel('Cumulative capacity\nby source [MW]')
plt.title('Cumulative capacity growth by energy source')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.legend(title='Energy source')
plt.tight_layout()
plt.show()

### CREATE A DAILY DATE-VALUE DATAFRAME

In [ ]:
# 1. Keep only necessary columns
df_ts = df_series[['Fecha de puesta en operación FPO', 'Capacidad acumulada [MW]']].copy()

# 2. Group by date keeping the maximum value if there are duplicates
df_ts = df_ts.groupby('Fecha de puesta en operación FPO').max()

# 3. Create date index in ascending order
df_ts = df_ts.sort_index()

# 4. Create a daily range from the minimum to maximum date
date_range = pd.date_range(start=df_ts.index.min(), end=df_ts.index.max(), freq='D')

# 5. Reindex to the daily range and fill missing values with the last known value
df_ts = df_ts.reindex(date_range).ffill()

# 6. Rename index to Fecha
df_ts.index.name = 'Fecha'

# ==============================
# 2. FIGURE: df_ts (daily series)
# ==============================
plt.figure(figsize=(5,3))
plt.plot(df_ts.index,
         df_ts['Capacidad acumulada [MW]'])
plt.xlabel('Date')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Daily cumulative capacity growth (df_ts)')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 3. FIGURE: Combined comparison
# ==============================
plt.figure(figsize=(5,3))
plt.plot(df_series['Fecha de puesta en operación FPO'],
         df_series['Capacidad acumulada [MW]'],
         label='Original data')
plt.plot(df_ts.index,
         df_ts['Capacidad acumulada [MW]'],
         label='Daily series')
plt.xlabel('Date')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Cumulative capacity curve comparison')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_solar = df_series[df_series['FUENTE DE ENERGIA'] == 'PLANTA SOLAR'].copy()
df_hydraulic = df_series[df_series['FUENTE DE ENERGIA'] == 'PLANTA HIDRÁULICA'].copy()
df_thermal = df_series[df_series['FUENTE DE ENERGIA'] == 'PLANTA TÉRMICA'].copy()

plt.figure(figsize=(5,3))
plt.plot(df_hydraulic['Fecha de puesta en operación FPO'], df_hydraulic['Capacidad acumulada [MW]'])
plt.plot(df_thermal['Fecha de puesta en operación FPO'], df_thermal['Capacidad acumulada [MW]'])
plt.plot(df_solar['Fecha de puesta en operación FPO'], df_solar['Capacidad acumulada [MW]'])
plt.xlabel('Date')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Cumulative capacity by energy source')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()